# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [2]:
from google.colab import drive
import pandas as pd
import json

drive.mount('/content/drive')
with open('/content/drive/MyDrive/nlp_ass3_drive/evidence.json', 'r') as f:
    evidence_json = json.load(f)

Mounted at /content/drive


In [3]:
from transformers import pipeline
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# classifier = pipeline("zero-shot-classification",
#                       model="facebook/bart-large-mnli",
#                       device=0)
BART_nli_model = AutoModelForSequenceClassification.from_pretrained('facebook/bart-large-mnli')
BART_tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large-mnli')
nli_model = BART_nli_model.to("cuda")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [37]:
import copy

def BART_batch_pipeline(premises_list, hypotheses_list, tokenizer, model, device):
    """
    Args:
        premises: 列表，例如 ["证据1", "证据2", ...]
        hypotheses: 列表，例如 ["声明1", "声明1", ...] # 注意长度要和 premises 一致
    """
    # 1. 使用 tokenizer 处理列表。必须加 padding 和 truncation，否则无法形成矩阵
    inputs = tokenizer(
        premises_list,
        hypotheses_list,
        return_tensors='pt',
        padding=True,           # 将短句子补齐到当前 batch 的最长长度
        truncation='only_first', # 优先截断 premise（证据），保留完整声明
        max_length=512
    ).to(device)

    # 2. 推理阶段建议开启 no_grad，节省显存并加快速度
    with torch.no_grad():
        # model(**inputs) 会自动处理 input_ids, attention_mask 等
        outputs = model(**inputs)
        logits = outputs.logits # 获取原始输出 [Batch_Size, 3]

    # 3. 在维度 1 (类别维度) 应用 Softmax
    probs = logits.softmax(dim=1)

    return probs
  # prob_contradiction = probs[0, 0].item()
  # prob_neutral       = probs[0, 1].item()
  # prob_entailment    = probs[0, 2].item()

def aggregate_to_four_classes_with_tuning(probs, nei_threshold, diff_threshold):
    """
    针对 CLIMATE-FEVER 任务设计的聚合逻辑

    Args:
        probs: [N, 3] 的 Tensor (0: REFUTES, 1: NEI, 2: SUPPORTS)
        nei_threshold (alpha): 中立证据占总量的比例阈值。超过此值则判定为 NEI
        diff_threshold (beta): 支持与反驳证据数量的相对差值阈值。小于此值判定为 DISPUTED
    """
    N = probs.shape[0]

    # 1. 拿到每一条证据的微观判决 (Micro-verdicts)
    micro_verdicts = probs.argmax(dim=1)

    # 2. 统计各类别出现的次数
    counts = torch.bincount(micro_verdicts, minlength=3)
    ref_count = counts[0].item()  # 反驳数量
    nei_count = counts[1].item()  # 中立数量
    sup_count = counts[2].item()  # 支持数量

    # --- 逻辑 A: 中立过滤 (旋钮 alpha) ---
    # 如果大部分证据都是 NEI，说明该声明缺乏明确的证据支撑
    if (nei_count / N) >= nei_threshold:
        return "NOT_ENOUGH_INFO"

    # --- 逻辑 B: 冲突判定 (旋钮 beta) ---
    decisive_count = sup_count + ref_count

    if sup_count > 0 and ref_count > 0:
        # 计算支持和反驳的相对差值
        # diff_ratio = |N_sup - N_ref| / (N_sup + N_ref)
        diff_ratio = abs(sup_count - ref_count) / decisive_count

        # 如果正反方势均力敌 (差值比例很小)，判定为 DISPUTED
        if diff_ratio <= diff_threshold:
            return "DISPUTED"

    return "SUPPORTS" if sup_count > ref_count else "REFUTES"

def run_prediction(nei_threshold,diff_threshold):

    """
    这里 model_pipeline 接收的就是你传进来的 classifier 对象
    """
    with open('/content/drive/MyDrive/nlp_ass3_drive/dev-claims.json', 'r') as dev_json:
      dev_json = json.load(dev_json)

    result_json = copy.deepcopy(dev_json)

    for claim_id, value in dev_json.items():
        claim_text = value['claim_text']
        evidence_ids = value['evidences']

        # 1. 提取真正的证据文本内容
        # 如果你还没写这个逻辑，记得先加载完整的 evidence.json
        evidence_texts = [evidence_json[eid] for eid in evidence_ids]

        # 2. 将声明文本复制成和证据数量一样多，保证一一对应
        claims_repeated = [claim_text] * len(evidence_texts)

        # 3. 调用你的批量 Pipeline
        # 传入：[证据1, 证据2...], [声明, 声明...]
        probs = BART_batch_pipeline(evidence_texts, claims_repeated, BART_tokenizer, nli_model, "cuda")

        # 此时 probs 是一个 [len(evidence_ids), 3] 的矩阵
        # print(f"Claim ID: {claim_id}, 得到矩阵形状: {probs.shape}")
        # v
        # 4. 这里就是你做均值和方差分析的地方了
        label = aggregate_to_four_classes_with_tuning(probs,nei_threshold,diff_threshold)
        result_json[claim_id]["claim_label"] = label
    return result_json


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [42]:
import json
import numpy as np


def load_json_file(path):
    """Load a JSON file and return a dict."""
    try:
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)
    except Exception as error:
        raise ValueError(f"Error loading json file: {path}") from error

    if not isinstance(data, dict):
        raise ValueError(f"JSON content must be an object/dict: {path}")
    return data


def validate_inputs(predictions, groundtruth):
    """Validate top-level evaluation inputs."""
    if not isinstance(predictions, dict):
        raise TypeError("predictions must be a dict")
    if not isinstance(groundtruth, dict):
        raise TypeError("groundtruth must be a dict")


def evaluate_from_data(predictions, groundtruth, show_each=False):
    """Core evaluation logic using in-memory JSON dicts."""
    validate_inputs(predictions, groundtruth)

    f_scores, accuracies = [], []

    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id in predictions and \
            "claim_label" in predictions[claim_id] and \
            "evidences" in predictions[claim_id]:

            instance_correct = 0.0
            if predictions[claim_id]["claim_label"] == claim["claim_label"]:
                instance_correct = 1.0

            evidence_correct = 0
            evidence_recall = 0.0
            evidence_precision = 0.0
            evidence_fscore = 0.0

            if isinstance(predictions[claim_id]["evidences"], list) and len(predictions[claim_id]["evidences"]) > 0:
                retrieved_ev = set(predictions[claim_id]["evidences"])
                for gt_ev in claim["evidences"]:
                    if gt_ev in retrieved_ev:
                        evidence_correct += 1

                if evidence_correct > 0:
                    evidence_recall = float(evidence_correct) / len(claim["evidences"])
                    evidence_precision = float(evidence_correct) / len(predictions[claim_id]["evidences"])
                    evidence_fscore = (2 * evidence_precision * evidence_recall) / (evidence_precision + evidence_recall)

            if show_each:
                print("groundtruth =", claim)
                print("predictions =", predictions[claim_id])
                print("instance accuracy =", instance_correct)
                print("evidence recall =", evidence_recall)
                print("evidence precision =", evidence_precision)
                print("evidence fscore =", evidence_fscore, "\n\n")

            accuracies.append(instance_correct)
            f_scores.append(evidence_fscore)

    mean_f = np.mean(f_scores if len(f_scores) > 0 else [0.0])
    mean_acc = np.mean(accuracies if len(accuracies) > 0 else [0.0])
    if mean_f == 0.0 and mean_acc == 0.0:
        hmean = 0.0
    else:
        hmean = (2 * mean_f * mean_acc) / (mean_f + mean_acc)

    return {
        "F": float(mean_f),
        "A": float(mean_acc),
        "H": float(hmean),
    }


def eval_func(predictions_path=None, groundtruth_path=None, predictions=None, groundtruth=None, show_each=False):
    """
    Wrapper entry point.
    - Use predictions_path/groundtruth_path to load from files.
    - Or pass predictions/groundtruth directly as dict.
    """
    try:
        if predictions is None:
            if predictions_path is None:
                raise ValueError("predictions_path or predictions must be provided")
            predictions = load_json_file(predictions_path)

        if groundtruth is None:
            if groundtruth_path is None:
                raise ValueError("groundtruth_path or groundtruth must be provided")
            groundtruth = load_json_file(groundtruth_path)
        scores = evaluate_from_data(predictions, groundtruth, show_each=show_each)
        print("Evidence Retrieval F-score (F)    =", scores["F"])
        print("Claim Classification Accuracy (A) =", scores["A"])
        print("Harmonic Mean of F and A          =", scores["H"])
        return scores
    except Exception as error:
        print("Error:", error)
        raise SystemExit

In [43]:
groundtruth_file_path = '/content/drive/MyDrive/nlp_ass3_drive/dev-claims.json'
nei_thresholds_list = [0.7]
diff_thresholds_list = [0.3]
for nei_threshold in nei_thresholds_list:
  for diff_threshold in diff_thresholds_list:
    predict_dic = run_prediction(nei_threshold,diff_threshold)
    print("nei =" + str(nei_threshold) + " diff =" + str(diff_threshold))
    eval_func(predictions=predict_dic, groundtruth_path = groundtruth_file_path)


nei =0.7 diff =0.4
Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.4025974025974026
Harmonic Mean of F and A          = 0.5740740740740741
nei =0.7 diff =0.5
Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.4025974025974026
Harmonic Mean of F and A          = 0.5740740740740741
nei =0.7 diff =0.6
Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.4025974025974026
Harmonic Mean of F and A          = 0.5740740740740741
nei =0.7 diff =0.7
Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.4025974025974026
Harmonic Mean of F and A          = 0.5740740740740741
nei =0.7 diff =0.8
Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.4025974025974026
Harmonic Mean of F and A          = 0.5740740740740741


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*